# 🧠 Session 01 — Introduction to Machine Learning (Part 2)

**Covers**: TL2 — Bias & Fairness in ML, Real-World Ethical Dilemmas, ML's Impact on Data Processing

**Duration**: 2 Hours | **Level**: 🟢 Beginner → 🟡 Intermediate → 🔴 Advanced

---

## 📋 What You'll Learn
1. What bias in AI is and why it matters
2. Real-world case studies of AI bias (Amazon, COMPAS, Google)
3. Types of bias: Data, Algorithmic, and Human
4. Fairness metrics: Demographic Parity, Equalized Odds
5. How to audit a model for bias using real data
6. Bias mitigation strategies
7. Responsible AI frameworks and practices

---

In [ ]:
# Required imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline
print('✅ All libraries loaded!')

---
## Part 6: Ethics, Bias, and Fairness in AI

### 🟢 What is Bias in AI?

**Bias** occurs when an algorithm systematically favors or disadvantages certain groups because the **training data reflects existing societal biases**.

### Real-World Case Studies

| Case | What Happened | Root Cause |
|------|---------------|-----------|
| **Amazon Hiring (2018)** | AI downgraded résumés with "women's" | 10 years of male-dominated hiring data |
| **COMPAS (2016)** | Falsely labeled Black defendants as high risk | Biased historical arrest data |
| **Google Photos (2015)** | Labeled Black users as "gorillas" | Insufficient training data diversity |
| **Healthcare (2019)** | Underserved Black patients due to cost proxy | Biased proxy variable |
| **Apple Card (2019)** | Men got higher credit limits | Opaque algorithm, no bias testing |

### 🟡 Types of Bias in ML

**Data Bias:**
- **Selection Bias** — Data doesn't represent the real population
- **Historical Bias** — Data reflects past discrimination
- **Measurement Bias** — Inconsistent data collection across groups
- **Label Bias** — Annotator prejudice in labeling

**Algorithmic Bias:**
- **Evaluation Bias** — Wrong metrics used for assessment
- **Aggregation Bias** — One model for groups that behave differently

**Human Bias:**
- **Confirmation Bias** — Seeking confirming evidence
- **Automation Bias** — Over-trusting AI decisions

---
## Hands-On: Bias Audit on Real Data

We'll use the **UCI Adult Census Income Dataset** (48,842 records) to detect gender and racial bias in an income prediction model.

In [ ]:
# ══════════════════════════════════════════
# STEP 1: Load UCI Adult Census Dataset
# ══════════════════════════════════════════
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
           'marital_status', 'occupation', 'relationship', 'race', 'sex',
           'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

print('Downloading UCI Adult Census dataset...')
df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)
df = df.dropna()

print(f'✅ Loaded {len(df):,} records')
print(f'\nIncome Distribution:')
print(df['income'].value_counts())
print(f'\nGender Distribution:')
print(df['sex'].value_counts())

In [ ]:
# ══════════════════════════════════════════
# STEP 2: Explore Bias in the Data
# ══════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploring Potential Bias in Adult Census Data', fontsize=14, fontweight='bold')

# Income by gender
gender_income = df.groupby(['sex', 'income']).size().unstack(fill_value=0)
gender_income_pct = gender_income.div(gender_income.sum(axis=1), axis=0)
gender_income_pct.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Income Distribution by Gender')
axes[0].set_ylabel('Proportion')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Income by race
race_high_income = df[df['income'] == '>50K'].groupby('race').size() / df.groupby('race').size()
race_high_income.sort_values().plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('High Income Rate by Race')
axes[1].set_xlabel('P(Income > $50K)')

# Education distribution by gender
axes[2].hist([df[df['sex']=='Male']['education_num'], 
              df[df['sex']=='Female']['education_num']], 
             bins=15, label=['Male', 'Female'], alpha=0.7)
axes[2].set_title('Education Level by Gender')
axes[2].set_xlabel('Education (years)')
axes[2].legend()

plt.tight_layout()
plt.show()

# Print stats
print('\nHigh income rate by gender:')
for sex in ['Male', 'Female']:
    rate = (df[df['sex']==sex]['income'] == '>50K').mean()
    print(f'  {sex}: {rate:.1%}')

print('\n⚠️ Note: These disparities exist in the REAL DATA.')
print('   A model trained on this data will likely reproduce these biases.')

In [ ]:
# ══════════════════════════════════════════
# STEP 3: Train Income Prediction Model
# ══════════════════════════════════════════
df_original = df.copy()

# Encode categorical variables
df_enc = df.copy()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

X = df_enc.drop('income', axis=1)
y = df_enc['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Keep track of protected attributes in test set
test_sex = df_original.loc[X_test.index, 'sex'].values
test_race = df_original.loc[X_test.index, 'race'].values

# Scale and train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print('Overall Model Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.2%}')
print(f'  Precision: {precision_score(y_test, y_pred):.2%}')
print(f'  Recall:    {recall_score(y_test, y_pred):.2%}')
print(f'  F1 Score:  {f1_score(y_test, y_pred):.2%}')

In [ ]:
# ══════════════════════════════════════════
# STEP 4: BIAS AUDIT — Gender Fairness
# ══════════════════════════════════════════
print('═' * 60)
print('  BIAS AUDIT: Gender Fairness Analysis')
print('═' * 60)

gender_metrics = {}
for gender in ['Male', 'Female']:
    mask = test_sex == gender
    g_true = y_test.values[mask]
    g_pred = y_pred[mask]
    
    acc = accuracy_score(g_true, g_pred)
    pos_rate = g_pred.mean()  # Demographic Parity
    
    tn, fp, fn, tp = confusion_matrix(g_true, g_pred, labels=[0,1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    gender_metrics[gender] = {
        'count': mask.sum(), 'accuracy': acc,
        'positive_rate': pos_rate, 'tpr': tpr, 'fpr': fpr
    }
    
    print(f'\n  {gender} ({mask.sum():,} samples):')
    print(f'    Accuracy:           {acc:.2%}')
    print(f'    Positive Pred Rate: {pos_rate:.2%}  ← Demographic Parity')
    print(f'    True Positive Rate: {tpr:.2%}  ← Equalized Odds')
    print(f'    False Positive Rate:{fpr:.2%}')

# Disparity check
dp_ratio = gender_metrics['Female']['positive_rate'] / max(gender_metrics['Male']['positive_rate'], 1e-10)
eo_diff = abs(gender_metrics['Male']['tpr'] - gender_metrics['Female']['tpr'])

print(f'\n  ─── Disparity Metrics ───')
print(f'  Demographic Parity Ratio: {dp_ratio:.3f}')
print(f'    (Should be 0.8–1.25 for fairness — "80% rule")')
print(f'    Status: {"✅ PASS" if 0.8 <= dp_ratio <= 1.25 else "❌ FAIL"}')
print(f'  Equalized Odds TPR Diff:  {eo_diff:.3f}')
print(f'    (Should be < 0.1 for fairness)')
print(f'    Status: {"✅ PASS" if eo_diff < 0.1 else "❌ FAIL"}')

In [ ]:
# ══════════════════════════════════════════
# STEP 5: BIAS AUDIT — Racial Fairness
# ══════════════════════════════════════════
print('═' * 60)
print('  BIAS AUDIT: Racial Fairness Analysis')
print('═' * 60)

race_metrics = {}
for race in df_original['race'].unique():
    mask = test_race == race
    if mask.sum() < 10:
        continue
    g_true = y_test.values[mask]
    g_pred = y_pred[mask]
    pos_rate = g_pred.mean()
    acc = accuracy_score(g_true, g_pred)
    race_metrics[race] = {'count': mask.sum(), 'positive_rate': pos_rate, 'accuracy': acc}
    print(f'  {race:25s} ({mask.sum():>5,}) | Pos Rate: {pos_rate:.2%} | Acc: {acc:.2%}')

In [ ]:
# ══════════════════════════════════════════
# STEP 6: Visualize Bias
# ══════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('AI Bias Audit — UCI Adult Census', fontsize=16, fontweight='bold')

# Gender positive rates
genders = list(gender_metrics.keys())
pos_rates = [gender_metrics[g]['positive_rate'] for g in genders]
bars = axes[0,0].bar(genders, pos_rates, color=['#3498db', '#e74c3c'], width=0.5)
axes[0,0].set_title('Demographic Parity: Positive Rate by Gender')
axes[0,0].set_ylabel('P(Predicted High Income)')
for bar, rate in zip(bars, pos_rates):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                   f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')

# Gender TPR/FPR
x = np.arange(len(genders))
tprs = [gender_metrics[g]['tpr'] for g in genders]
fprs = [gender_metrics[g]['fpr'] for g in genders]
axes[0,1].bar(x-0.15, tprs, 0.3, label='TPR', color='#2ecc71')
axes[0,1].bar(x+0.15, fprs, 0.3, label='FPR', color='#e74c3c')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(genders)
axes[0,1].set_title('Equalized Odds: TPR & FPR by Gender')
axes[0,1].legend()

# Race positive rates
races = list(race_metrics.keys())
race_rates = [race_metrics[r]['positive_rate'] for r in races]
axes[1,0].barh(races, race_rates, color=plt.cm.Set2(np.linspace(0,1,len(races))))
axes[1,0].set_title('Positive Rate by Race')
axes[1,0].set_xlabel('P(Predicted High Income)')

# Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns).nlargest(10)
colors = ['#e74c3c' if f in ['sex','race','native_country'] else '#3498db' for f in importances.index]
importances.plot(kind='barh', ax=axes[1,1], color=colors)
axes[1,1].set_title('Top 10 Features (Red = Protected Attribute)')

plt.tight_layout()
plt.show()

### 🟡 Fairness Metrics Summary

| Metric | Definition | Goal |
|--------|-----------|------|
| **Demographic Parity** | P(Ŷ=1\|A=0) = P(Ŷ=1\|A=1) | Equal positive prediction rates |
| **Equalized Odds** | Equal TPR and FPR across groups | Equal error rates |
| **Predictive Parity** | Equal precision across groups | Equal trust |

> ⚠️ **Impossibility Theorem (Chouldechova, 2017)**: You CANNOT satisfy all fairness metrics simultaneously when base rates differ. Fairness is a **design choice**.

In [ ]:
# ══════════════════════════════════════════
# STEP 7: Bias Mitigation — Remove Protected Features
# ══════════════════════════════════════════
print('═' * 60)
print('  BIAS MITIGATION: Removing Protected Attributes')
print('═' * 60)

protected = ['sex', 'race', 'native_country']
X_fair = X.drop(columns=protected)

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y)

sc_f = StandardScaler()
model_fair = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model_fair.fit(sc_f.fit_transform(X_tr_f), y_tr_f)
y_pred_f = model_fair.predict(sc_f.transform(X_te_f))

# Compare
orig_acc = accuracy_score(y_test, y_pred)
fair_acc = accuracy_score(y_te_f, y_pred_f)
orig_f1 = f1_score(y_test, y_pred)
fair_f1 = f1_score(y_te_f, y_pred_f)

print(f'\n  {"Metric":<20s} {"Original":>12s} {"Fair Model":>12s} {"Change":>10s}')
print(f'  {"─"*55}')
print(f'  {"Accuracy":<20s} {orig_acc:>12.2%} {fair_acc:>12.2%} {fair_acc-orig_acc:>+10.2%}')
print(f'  {"F1 Score":<20s} {orig_f1:>12.2%} {fair_f1:>12.2%} {fair_f1-orig_f1:>+10.2%}')

# Check fairness of new model
test_sex_f = df_original.loc[X_te_f.index, 'sex'].values
for gender in ['Male', 'Female']:
    mask = test_sex_f == gender
    pos_rate = y_pred_f[mask].mean()
    print(f'\n  {gender} Positive Rate: {pos_rate:.2%}')

print('\n⚠️ Note: Removing protected features helps but doesn\'t eliminate bias.')
print('   Proxy variables (e.g., occupation, marital status) can still encode bias.')

---
## Part 7: Responsible AI Practices

| Principle | What It Means | Action |
|-----------|---------------|--------|
| **Transparent** | Explain decisions | Use SHAP/LIME for explainability |
| **Fair** | No discrimination | Audit for bias, use fairness constraints |
| **Accountable** | Clear ownership | Document models, create model cards |
| **Private** | Protect personal data | GDPR compliance, anonymization |
| **Safe** | Don't cause harm | Human oversight, gradual deployment |

### Ethical Frameworks

| Framework | Key Requirements |
|-----------|------------------|
| **EU AI Act (2024)** | Risk-based classification, mandatory audits for high-risk |
| **IEEE Ethically Aligned Design** | Transparency, accountability |
| **OECD AI Principles** | Human-centered, inclusive growth |

### ML's Impact on Data Processing

ML transforms data processing through:
- **Automation**: Replacing manual analysis of massive datasets
- **Pattern Discovery**: Finding insights humans would miss
- **Real-Time Processing**: Instant fraud detection, recommendations
- **Scalability**: Handling billions of records seamlessly

But it also introduces **ethical risks** — biased automation at scale can harm millions.

In [ ]:
# ══════════════════════════════════════════
# Responsible AI Checklist Function
# ══════════════════════════════════════════

def responsible_ai_audit(model, X_test, y_test, y_pred, protected_col_values, col_name):
    """Run a basic Responsible AI audit."""
    print(f'\n{"═"*50}')
    print(f'  RESPONSIBLE AI AUDIT: {col_name}')
    print(f'{"═"*50}')
    
    # Overall metrics
    print(f'\n  Overall Accuracy: {accuracy_score(y_test, y_pred):.2%}')
    
    # Per-group analysis  
    for group in np.unique(protected_col_values):
        mask = protected_col_values == group
        if mask.sum() < 10: continue
        acc = accuracy_score(y_test.values[mask], y_pred[mask])
        pos_rate = y_pred[mask].mean()
        print(f'  {group:20s}: Acc={acc:.2%}, PosRate={pos_rate:.2%} ({mask.sum():,} samples)')
    
    # Feature importance check
    if hasattr(model, 'feature_importances_'):
        top5 = pd.Series(model.feature_importances_, 
                         index=X_test.columns if hasattr(X_test, 'columns') else range(X_test.shape[1])
                        ).nlargest(5)
        print(f'\n  Top 5 Features:')
        for f, imp in top5.items():
            print(f'    {f}: {imp:.4f}')

# Run the audit
responsible_ai_audit(model, X_test, y_test, y_pred, test_sex, 'Gender')
responsible_ai_audit(model, X_test, y_test, y_pred, test_race, 'Race')

print('\n✅ Always run this audit before deploying any model!')

---
## ✅ Session 01 Complete!

### Key Takeaways

**From TL1 (Part 1):**
1. ML learns patterns from data instead of explicit rules
2. 3 types: Supervised, Unsupervised, Reinforcement Learning
3. Every ML project follows: Define → Collect → Prepare → Train → Evaluate → Deploy
4. ML is used across every industry

**From TL2 (Part 2):**
1. AI bias is REAL and has caused harm (Amazon, COMPAS, etc.)
2. Bias comes from data, algorithms, AND humans
3. Fairness metrics: Demographic Parity, Equalized Odds, Predictive Parity
4. You CANNOT satisfy all fairness metrics simultaneously
5. Always audit models for bias before deployment
6. Responsible AI = Transparent + Fair + Accountable + Private + Safe

### 📊 Portfolio Task
Create a **Responsible AI Audit Report** — see `portfolio/portfolio_component.md`

### ✍️ Exercises
Complete all exercises in `exercises/exercises.md` (Beginner through Advanced)

---
*Next Session: [Session 02 — Advanced Clustering Algorithms](../Session_02_Advanced_Clustering/)*